In [ ]:
import os
import pickle
import numpy as np
import torch
import torch.nn.functional as F
import mrcfile
from tqdm import tqdm

print('torch:', torch.__version__)
print('CUDA disponibile:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Working dir:', os.getcwd())

In [ ]:
# ── Path dataset ─────────────────────────────────────────────────────────────
DATASET_ROOT  = 'IgG-1D/IgG-1D'
POSES_PATH    = f'{DATASET_ROOT}/combined_poses.pkl'
VOLS_DIR      = f'{DATASET_ROOT}/vols/128_org'
OUT_DIR       = f'{DATASET_ROOT}/clean_projections'
N_VOLUMES     = 100
POSES_PER_VOL = 1000

os.makedirs(OUT_DIR, exist_ok=True)

# ── Ispezione struttura combined_poses.pkl ───────────────────────────────────
with open(POSES_PATH, 'rb') as f:
    _tmp = pickle.load(f)

print('Tipo oggetto:', type(_tmp))
for k, v in enumerate(_tmp):
    arr = np.array(v)
    print(f'  [{k}] dtype={arr.dtype}  shape={arr.shape}')

del _tmp

In [ ]:
def generate_clean_projections(vol_path, poses_tuple, batch_size=64, device='cuda'):
    with mrcfile.open(vol_path, mode='r') as mrc:
        vol_data = mrc.data.copy()

    vol_data = (vol_data - np.mean(vol_data)) / np.std(vol_data)

    vol_tensor = torch.tensor(vol_data, dtype=torch.float32, device=device)
    vol_tensor = vol_tensor.unsqueeze(0).unsqueeze(0)

    D = vol_tensor.shape[-1]
    projections_list = []

    rotations    = torch.tensor(np.array(poses_tuple[0], dtype=np.float32), dtype=torch.float32)
    translations = torch.tensor(np.array(poses_tuple[1], dtype=np.float32), dtype=torch.float32)
    N = rotations.shape[0]

    for i in tqdm(range(0, N, batch_size), desc='Proiettando', leave=False):
        end_idx = min(i + batch_size, N)
        B = end_idx - i

        R_batch = rotations[i:end_idx]
        t_batch = translations[i:end_idx]

        R_inv  = R_batch.transpose(1, 2)
        t_norm = -t_batch / (D / 2.0)
        t_3d   = torch.cat([t_norm, torch.zeros(B, 1)], dim=1).unsqueeze(2)
        theta  = torch.cat([R_inv, t_3d], dim=2).to(device)

        grid        = F.affine_grid(theta, size=(B, 1, D, D, D), align_corners=False)
        rotated_vol = F.grid_sample(vol_tensor.expand(B, -1, -1, -1, -1), grid, align_corners=False)
        proj_2d     = rotated_vol.sum(dim=2)
        projections_list.append(proj_2d.cpu().numpy())

    return np.concatenate(projections_list, axis=0)

In [ ]:
# ── Caricamento pose ─────────────────────────────────────────────────────────
with open(POSES_PATH, 'rb') as f:
    poses_data = pickle.load(f)

rots_all  = np.array(poses_data[0])   # (100000, 3, 3) float32
trans_all = np.array(poses_data[1])   # (100000, 2)    float64 -> cast in funzione

# Auto-detect layout: flat (100000,3,3) oppure nested (100,1000,3,3)
if rots_all.ndim == 4 and rots_all.shape[:2] == (N_VOLUMES, POSES_PER_VOL):
    print(f'Layout NESTED: {rots_all.shape}')
    def _get_poses(i):
        return rots_all[i], trans_all[i]
elif rots_all.ndim == 3 and rots_all.shape[0] == N_VOLUMES * POSES_PER_VOL:
    print(f'Layout FLAT: {rots_all.shape}')
    def _get_poses(i):
        s = i * POSES_PER_VOL
        return rots_all[s:s + POSES_PER_VOL], trans_all[s:s + POSES_PER_VOL]
else:
    raise ValueError(
        f'Struttura pose inattesa: rots shape={rots_all.shape}. '
        f'Atteso (100000,3,3) flat o (100,1000,3,3) nested.'
    )

# ── Loop 100 conformazioni ───────────────────────────────────────────────────
for i in tqdm(range(N_VOLUMES), desc='Conformazioni', unit='vol'):
    vol_path = f'{VOLS_DIR}/{i:03d}.mrc'
    out_path = f'{OUT_DIR}/clean_2d_targets_{i:03d}.npy'

    clean_projs = generate_clean_projections(vol_path, _get_poses(i))

    np.save(out_path, clean_projs)
    tqdm.write(f'[{i:03d}] {out_path} salvato -- shape {clean_projs.shape}')

In [ ]:
# ── Verifica visiva del primo output ─────────────────────────────────────────
import matplotlib.pyplot as plt

sample_projs = np.load(f'{OUT_DIR}/clean_2d_targets_000.npy')  # (1000, 1, 128, 128)
print('Shape output:', sample_projs.shape)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for idx, ax in enumerate(axes.flat):
    ax.imshow(sample_projs[idx, 0], cmap='gray')
    ax.set_title(f'Proj {idx}')
    ax.axis('off')
plt.suptitle('Clean 2D projections -- conformazione 000', fontsize=12)
plt.tight_layout()
plt.show()